# 04 — Train fold 2: DINOv2-S slot model (family 2)

GPU, Internet ON. Attach the competition, both cache datasets made from the 02 shard outputs (knee-cache-v3-shard0/1, latest version: 18 files of ~1 GB each), the four public report-label datasets, your reader_labels dataset and the 01_labels output (compi-label-generation, for the Flash-Lite labels). Inputs are detected automatically. Run all cells and save the executed output. All folds of a family must use identical inputs and settings.

## 0 · Config

In [ ]:
# Training requires Internet for pretrained weights and this small split helper.
import importlib.util, subprocess, sys
if importlib.util.find_spec('iterstrat') is None:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','iterative-stratification==0.1.9'])
# Keep the preinstalled CUDA torch/torchvision stack; match inference's timm.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'timm==1.0.29'])

In [ ]:
# Run cells in order; no working-directory changes are needed.

In [ ]:
# Run cells in order; no working-directory changes are needed.

In [ ]:
import os, gc, copy, math, time, json, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# ---- inputs ------------------------------------------------------------------
"""Bounded discovery of attached Kaggle artifacts; embedded in launch notebooks."""
import fnmatch, os
from pathlib import Path


DICOM_DIRS = ('train_series', 'test_series')   # competition image trees: never listed


def attached_files(name, root='/kaggle/input', max_depth=5):
    # Kaggle mounts datasets and notebook outputs at different depths; artifacts live in
    # folders 1-5 levels below the root. The DICOM trees are pruned: listing them took
    # ~2 min on local disk and over an hour on the TPU VM's network-mounted inputs.
    root = Path(root)
    found = set()
    for path, dirs, files in os.walk(root):
        depth = len(Path(path).relative_to(root).parts)
        dirs[:] = [d for d in dirs if d not in DICOM_DIRS] if depth < max_depth else []
        if depth >= 1:
            found.update(Path(path) / f for f in fnmatch.filter(files, name) if not f.startswith('.'))
    return sorted(found)


def one_input(name, override='', root='/kaggle/input'):
    if override:
        path = Path(override)
        if not path.is_file():
            raise FileNotFoundError(f'Configured input does not exist: {path}')
        return str(path)
    matches = attached_files(name, root)
    if len(matches) != 1:
        raise RuntimeError(f'Attach exactly one intended {name} input or set its path explicitly. '
                           f'Found: {[str(p) for p in matches]}')
    print(f'{name}: {matches[0]}')
    return str(matches[0])


def fold_inputs(expected, root='/kaggle/input', per_fold=1):
    # per_fold > 1 when several training runs (e.g. DINOv2 and DINOv3 arms) are attached.
    paths = []
    for fold in expected:
        matches = attached_files(f'fold_{fold}.pt', root)
        if len(matches) != per_fold:
            raise RuntimeError(f'Attach exactly {per_fold} fold_{fold}.pt input(s) '
                               f'({"exactly one" if per_fold == 1 else "one per training run"}). '
                               f'Found: {[str(p) for p in matches]}')
        paths.extend(str(p) for p in matches)
    for path in paths: print(f'checkpoint: {path}')
    return paths


def checkpoint_inputs(root='/kaggle/input'):
    # Every attached fold_N.pt is an ensemble member; families are resolved from each
    # checkpoint's recipe, and duplicates of one family-fold are rejected by the caller.
    paths = [str(p) for p in attached_files('fold_*.pt', root)]
    if not paths:
        raise RuntimeError('Attach at least one training output containing fold_N.pt')
    for path in paths: print(f'checkpoint: {path}')
    return paths

# Cache shards and label tables are discovered under /kaggle/input. Set CACHE_DIR to a
# comma-separated list of shard folders only if the discovery is ambiguous.
INPUT_ROOT = os.environ.get("KNEE_INPUT_ROOT", "/kaggle/input")   # other hosts: where inputs live
CACHE_DIR = ','.join(str(p.parent) for p in attached_files('cache_meta.json', INPUT_ROOT))
assert CACHE_DIR, 'Attach the 02_cache shard outputs'
LABEL_ROOT = INPUT_ROOT
COMP_DIR  = os.environ.get("KNEE_COMP_DIR") or next(
    (d for d in ("/kaggle/input/competitions/rsna-knee-abnormality-detection", "/kaggle/input/rsna-knee-abnormality-detection")
     if os.path.isdir(d)), "/kaggle/input/competitions/rsna-knee-abnormality-detection")
OUT       = os.environ.get("KNEE_OUT", "/kaggle/working")   # one folder per family in a session
# -----------------------------------------------------------------------------
# Several cache shards may be attached; a comma-separated list is accepted too.
CACHE_DIRS = [d for d in CACHE_DIR.split(',') if d]

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]

_metas = [json.load(open(os.path.join(d, 'cache_meta.json'))) for d in CACHE_DIRS]
_recipe_only = lambda m: {k: v for k, v in m.items() if k not in ('shard', 'n_shards')}
assert all(_recipe_only(m) == _recipe_only(_metas[0]) for m in _metas), 'Cache shards come from different recipes'
CACHE_META = _recipe_only(_metas[0])
_shards = sorted(m.get('shard', 0) for m in _metas)
assert _shards == list(range(_metas[0].get('n_shards', 1))), f'Attach every cache shard exactly once (found {_shards})'
assert CACHE_META.get('version') in ('knee-2c-v2', 'knee-2c-v3', 'knee-2c-v4', 'knee-2c-v5'), 'Rebuild the step-2c cache with current preprocessing'
PLANES = CACHE_META['slots']
CACHE_IMG = CACHE_META['img']
N_SLICES = CACHE_META['n_slices']

# ---- model recipe -------------------------------------------------------------
# Eight self-trained families, chosen by PRESET (one launcher set per family):
# reader_convnext: public 2.5D study reader (0.929 alone). ConvNeXt-T on windows of three
#   adjacent slices, slot and stack-position tokens, two transformer layers, attention
#   pooling per finding, EMA weights, warmup then cosine.
# reader_student: the ConvNeXt reader trained on report labels mixed with our teachers' OOF predictions.
# reader_student2: round 2, whose teacher also includes the round-1 students.
# reader_caformer: the same reader on CAFormer-S18 (ImageNet-22k), for backbone diversity.
# reader_coatnet: the same reader on CoAtNet-RMLP-2 at 384 px, the backbone of the public
#   Raptor single model (0.928 alone). Gradient checkpointing keeps it inside a T4.
# slot_dinov2: the public DINOv2 member (one spread 3-slice image per slot, last six
#   blocks fine-tuned, per-finding query head).
# radimagenet: frozen official RadImageNet ResNet-50 slice tokens with a per-finding query
#   head (public E9/E11 arm); encoded once, all folds in one run (rad_runtime.py).
# kneenet_b0: the earlier EfficientNet-b0 model, kept for comparison.
PRESET = "slot_dinov2"
PRESETS = {
    'reader_convnext': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                            epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                            pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                            clip=3., grad_checkpoint=False),
    # Noisy student: the same ConvNeXt reader recipe trained on targets mixed with the out-of-fold
    # predictions of our own readers (teacher_targets.csv). Agreement-gated: the teacher share runs
    # from 0.5 where the report readers agree confidently to 0.9 where they are split or silent.
    # Slot dropout hides whole MRI sequences in training (test sites differ in protocol).
    'reader_student': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                           epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                           pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                           clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # Round 2: the same student recipe, whose teacher also averages the round-1 students' OOF.
    # 11 epochs: pseudo-validation plateaus by epochs 10-12, and a two-fold session then fits
    # in under two GPU hours.
    'reader_student2': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                            epochs=11, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                            pct_start=1/11, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                            clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,
                            teacher_families={'reader_convnext': 1.0, 'reader_caformer': 1.0, 'reader_student': 1.0}),
    # Same reader, a different backbone family for ensemble diversity (convolution stages, then
    # attention; ImageNet-22k). ~23% more activation memory per image than ConvNeXt-T, whose
    # batch 4 peaked at 10.9 GB on a T4: half the micro-batch, same effective batch.
    'reader_caformer': dict(arch='reader25d', backbone='caformer_s18.sail_in22k_ft_in1k', img=CACHE_IMG,
                            epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                            pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                            clip=3., grad_checkpoint=False),
    # Round-1 student recipe and teacher on a ResNet-50 (batch-norm CNN) for ensemble diversity:
    # LB 2026-10-09/10: more ConvNeXt students/seeds added ~0.001; round-2 teachers did not help.
    # Half the micro-batch (activation memory), same effective batch.
    'reader_student_r50': dict(arch='reader25d', backbone='resnet50.a1_in1k', img=CACHE_IMG,
                               epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # Report-grounded student: the round-1 student recipe and teacher plus an auxiliary loss that
    # regresses each study's report embedding (scripts/report_embeddings.py; attach the
    # knee-report-embeddings dataset) from the images, at this weight relative to the label BCE.
    'reader_student_rep': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                               epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,
                               report_aux=.2),
    # Round-1 student recipe and teacher on CAFormer-S18 (in22k): its readers (session J) beat the
    # ConvNeXt readers on pseudo-validation (AUC 0.895/0.897 vs 0.893/0.891, folds 0/1); ResNet-50
    # students fell far behind (0.915 vs 0.963), so diversity comes from this family instead.
    'reader_student_caf': dict(arch='reader25d', backbone='caformer_s18.sail_in22k_ft_in1k', img=CACHE_IMG,
                               epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                               pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                               clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # Round-1 student recipe and teacher on ConvNeXt-Base (ImageNet-22k), ~3x ConvNeXt-T compute:
    # trained on the TPU (08_tpu_student_cnxb), where the long run does not hold up GPU sessions.
    'reader_student_cnxb': dict(arch='reader25d', backbone='convnext_base.fb_in22k_ft_in1k', img=CACHE_IMG,
                                epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # The round-1 ConvNeXt student recipe and teacher on the knee-2c-v4 cache (six v3 slots + two
    # 90 mm local slots around the header-geometry joint anchor); a separate family name so its
    # LB comparison with the v3 students (0.935 on folds 0-1) stays clean.
    'reader_student_v4': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                              epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                              pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                              clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # ConvNeXt-Base student on knee-2c-v4 (TPU): LB 2026-10-12 showed both gains separately,
    # ConvNeXt-B v3 x5 = 0.938 vs ConvNeXt-T v3 x6 = 0.936, and v4 folds 0-1 = 0.938 vs v3 = 0.935.
    'reader_student_cnxb_v4': dict(arch='reader25d', backbone='convnext_base.fb_in22k_ft_in1k', img=CACHE_IMG,
                                   epochs=14, batch=4, accum=2, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                   pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                   clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # The ConvNeXt student on knee-2c-v5 (288 px, 10 windows per series; TPU).
    # 288 px x 8 slots: half the T4 micro-batch of v4 (activation memory), same effective batch of 8.
    'reader_student_v5': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                              epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                              pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                              clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    # ConvNeXt-Base and CAFormer-S18 students on knee-2c-v5 (TPU / GPU after the 2026-10-17 quota reset).
    # On a T4: one study per micro-batch (48 windows at 288 px) with gradient checkpointing; the TPU
    # runtime ignores both (8 studies per step across the chips, no checkpointing). 8 epochs: at 50-60
    # min per epoch on a T4, 14 would exceed the 12 h session (2026-10-14); ConvNeXt-B at 288 px
    # reached val BCE 0.375 / AUC 0.954 by epoch 3.
    'reader_student_cnxb_v5': dict(arch='reader25d', backbone='convnext_base.fb_in22k_ft_in1k', img=CACHE_IMG,
                                   epochs=8, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                   pct_start=1/8, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                   clip=3., grad_checkpoint=True, teacher=(.5, .9), slot_dropout=.2),
    # v5 students taught partly by heliosli's compact model (its out-of-fold predictions on the training
    # studies, 10_compact_oof_*): teacher = our ConvNeXt readers (sessions A-F, the Z5/B5 teacher; no CAFormer
    # reader outputs are attached) and the compact model at 1.0 each, i.e. half ours and half compact, its
    # per-finding ranking put on our teachers' value distribution (its report targets use other label
    # definitions, e.g. synovitis mean 0.54 vs 0.30);
    # otherwise the ConvNeXt-T / -B v5 recipes.
    'reader_student_ct_v5': dict(arch='reader25d', backbone='convnext_tiny.in12k_ft_in1k', img=CACHE_IMG,
                                 epochs=14, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                 pct_start=1/14, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                 clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2,
                                 teacher_families={'reader_convnext': 1.0, 'reader_compact': 1.0},
                                 teacher_quantile_map=('reader_compact',)),
    'reader_student_cnxb_ct_v5': dict(arch='reader25d', backbone='convnext_base.fb_in22k_ft_in1k', img=CACHE_IMG,
                                      epochs=8, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                      pct_start=1/8, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                      clip=3., grad_checkpoint=True, teacher=(.5, .9), slot_dropout=.2,
                                      teacher_families={'reader_convnext': 1.0, 'reader_compact': 1.0},
                                      teacher_quantile_map=('reader_compact',)),
    'reader_student_caf_v5': dict(arch='reader25d', backbone='caformer_s18.sail_in22k_ft_in1k', img=CACHE_IMG,
                                  epochs=7, batch=1, accum=8, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                                  pct_start=1/7, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                                  clip=3., grad_checkpoint=False, teacher=(.5, .9), slot_dropout=.2),
    'reader_coatnet': dict(arch='reader25d', backbone='coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k', img=384,
                           epochs=10, batch=2, accum=4, lr=5e-4, encoder_lr_scale=.2, wd=.02,
                           pct_start=1/10, dropout=.2, unfreeze_last=None, head_hidden=384, ema=.998,
                           clip=3., grad_checkpoint=True),
    'slot_dinov2': dict(arch='slot_query', backbone='vit_small_patch14_dinov2.lvd142m', img=224, epochs=10,
                        batch=4, accum=2, lr=1e-3, encoder_lr_scale=8e-3, wd=.02, pct_start=.15, dropout=.2,
                        unfreeze_last=6, head_hidden=256, ema=0., clip=5., grad_checkpoint=False),
    'radimagenet': dict(arch='radhead', backbone='radimagenet_resnet50', img=224, epochs=24, batch=48,
                        accum=1, lr=2e-4, encoder_lr_scale=0., wd=3e-3, pct_start=0., dropout=.15,
                        unfreeze_last=None, head_hidden=512, ema=0., clip=1., grad_checkpoint=False),
    'kneenet_b0': dict(arch='kneenet', backbone='tf_efficientnet_b0.ns_jft_in1k', img=CACHE_IMG, epochs=6,
                       batch=4, accum=1, lr=3e-4, encoder_lr_scale=.3, wd=1e-2, pct_start=.1, dropout=.4,
                       unfreeze_last=None, head_hidden=None, ema=0., clip=5., grad_checkpoint=False),
}
_preset = PRESETS[PRESET]
_needs = {'_v4': 'knee-2c-v4', '_v5': 'knee-2c-v5'}.get(PRESET[-3:])
assert _needs is None or CACHE_META.get('version') == _needs, f'Attach the {_needs} cache outputs'
ARCH = _preset['arch']
BACKBONE = os.environ.get("KNEE_BACKBONE", _preset['backbone'])
IMG = _preset['img']              # model input size; the cache is resized on the GPU if different
EPOCHS = _preset['epochs']
BATCH = _preset['batch']          # studies per micro-batch
ACCUM_STEPS = _preset['accum']    # effective batch = BATCH * ACCUM_STEPS studies
LR = _preset['lr']                # head learning rate
ENCODER_LR_SCALE = _preset['encoder_lr_scale']   # encoder rate = LR * this
WD = _preset['wd']
PCT_START = _preset['pct_start']  # warmup share of the one-cycle schedule
DROPOUT = _preset['dropout']
UNFREEZE_LAST = _preset['unfreeze_last']
HEAD_HIDDEN = _preset['head_hidden']
EMA_DECAY = _preset['ema']        # 0 disables; EMA weights are validated and saved
CLIP = _preset['clip']
GRAD_CHECKPOINT = _preset['grad_checkpoint']
TEACHER_WEIGHT = _preset.get('teacher', 0.0)   # constant or (lo, hi) gate; attach teacher_targets.csv
SLOT_DROPOUT = _preset.get('slot_dropout', 0.0) # reader25d: chance of hiding each present series in training
TEACHER_FAMILY_WEIGHTS = _preset.get('teacher_families')   # None: label_builder.TEACHER_FAMILIES
TEACHER_QUANTILE_MAP = _preset.get('teacher_quantile_map', ())   # outside teachers put on our teachers' scale
REPORT_AUX = _preset.get('report_aux', 0.0)    # weight of the report-embedding auxiliary loss; 0 disables
TRAIN_WINDOWS = 6                 # reader25d: windows encoded per series per step (of 8)
# Gold studies train in every fold at this per-cell weight (decision 2026-10-07). Their
# audit AUC is then in-sample and is printed as such; selection never reads gold.
GOLD_WEIGHT = 3.0
READER_WEIGHTS = {}               # e.g. {'extra_labels_sol_v2.csv': 2.0}; default 1 each
# Every reader the run must find; a missing attachment stops before any GPU time is spent.
REQUIRED_READERS = ['report_labels_v2.csv', 'llm_labels_v4_blend.csv', 'labels_llm_gpt56sol.csv',
                    'qwen_knee_weak_labels.csv', 'extra_labels_sol_v2.csv',
                    'extra_labels_sonnet_legacy.csv', 'pseudo_google_gemini*flash*.jsonl']
NORMALIZATION = 'pretrained'
N_FOLDS = 5
FOLDS_TO_RUN = [2]
GROUP_COL = 'report_group'        # folds keep duplicate reports together
RESUME = False  # copy both fold_N.pt and last_fold_N.pt to OUT before enabling
T_TRAIN     = 10       # kneenet only: contiguous slices sampled per plane per step
SEED        = int(os.environ.get("KNEE_SEED", 42))   # sessions may train extra seeds of a family
NUM_WORKERS = int(os.environ.get("KNEE_WORKERS", 4))   # Kaggle gives 4 cores; two-GPU sessions use 2 per job

def write_atomic(path, content):
    # Two jobs in one session may write the same small file (folds, config); replace it whole.
    # content: text, or a function that writes to the temporary path it is given.
    temp = f'{path}.{os.getpid()}.tmp'
    if callable(content):
        content(temp)
    else:
        with open(temp, 'w') as handle: handle.write(content)
    os.replace(temp, path)


os.makedirs(OUT, exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
torch.backends.cudnn.benchmark = True          # shapes are fixed -> let cuDNN autotune

print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")
if DEV == "cuda" and torch.cuda.device_count() > 1:
    print(f"note: {torch.cuda.device_count()} GPUs visible, using 1 by design "
          f"(DataParallel benchmarked 13x slower here)")
if ARCH == 'kneenet' and BACKBONE.startswith('vit_'):
    BATCH, ACCUM_STEPS = 1, 4  # effective batch 4; lower activation memory
print(f"recipe: {PRESET} ({ARCH}) | {BACKBONE} @ {IMG}px | {CACHE_META['version']} {len(PLANES)} slots x "
      f"{N_SLICES} slices x {CACHE_IMG}px in {len(CACHE_DIRS)} shard(s) | {EPOCHS} epochs, "
      f"{BATCH}x{ACCUM_STEPS} studies/step | gold weight {GOLD_WEIGHT}")
assert DEV == "cuda", "Enable a GPU before starting training"

# Optional interrupted-fold recovery: attach that fold's previous saved output.
RESUME_DIR = ""
if RESUME_DIR:
    import shutil
    for fold in FOLDS_TO_RUN:
        for filename in (f'fold_{fold}.pt', f'last_fold_{fold}.pt'):
            previous = Path(RESUME_DIR) / filename
            assert previous.is_file(), previous
            destination = Path(OUT) / filename
            assert not destination.exists(), f'Restart the session before copying {destination}'
            shutil.copy2(previous, destination)
    RESUME = True

## 1 · Data

In [ ]:
"""Soft training targets from report-label readers; embedded in training.ipynb."""
import os, re, json, fnmatch, hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# Public report-label tables (rules allow public external data). File name -> suffix of
# the twelve target columns. The mean of these four reached 0.896 macro AUC on the gold
# studies in the public 2.5D reader's own measurement.
PUBLIC_LABEL_TABLES = {
    'report_labels_v2.csv': '',            # pilkwang/rsna-knee-llm-labels
    'llm_labels_v4_blend.csv': '',         # stevenleehans/rsna-knee-llm-report-labels
    'labels_llm_gpt56sol.csv': '',         # lixin73/rsna-knee-llm-report-labels-sol56
    'qwen_knee_weak_labels.csv': '__label',  # laymond/rsna-knee-abnormality-qwen3-8b-weak-labels
}
# Our own readers:
#  - extra_labels_*.csv from scripts/export_reader_labels.py (GPT Sol v2, Sonnet legacy);
#  - the Flash member file written by our 01_labels notebook (its labels/ folder), read
#    directly so Flash counts once. That notebook's consensus train_targets.csv is not read:
#    it already mixes Flash with Sonnet.
OWN_PREFIX = 'extra_labels'
OWN_JSONL_READERS = {'pseudo_google_gemini*flash*.jsonl': 'legacy'}

# State -> probability that the finding is positive. Legacy is the 01_labels notebook's own
# HARD map; v2 states follow the knee-report-labels-v2 protocol. Unsure abstains (NaN)
# under legacy; under v2 it marks an affirmative finding of undecidable eligibility.
STATE_P = {
    'legacy': {'definite': 1.0, 'probable': .75, 'possible': .35, 'absent': 0.0},
    'v2': {'definite': 1.0, 'unsure': .5, 'absent': 0.0},
}
STATE_KEYS = dict(zip(
    ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
     "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"],
    ["acl", "mcl", "medial_meniscus", "lateral_meniscus", "medial_oa", "lateral_oa",
     "pf_oa", "effusion", "synovitis", "bakers", "contusion", "fracture"]))


def iter_jsonl(path):
    # One historical shard was written with literal "\n" separators; split on both.
    with open(path) as handle:
        for chunk in re.split(r'\n|\\n', handle.read()):
            if chunk.strip():
                yield json.loads(chunk)


def states_frame(records, label_cols, protocol):
    mapping = STATE_P[protocol]
    rows = [{'StudyInstanceUID': str(r['StudyInstanceUID']),
             **{c: mapping.get(str(r.get(STATE_KEYS.get(c, c), '')).lower()) for c in label_cols}}
            for r in records]
    frame = pd.DataFrame(rows, columns=['StudyInstanceUID', *label_cols])
    return frame.drop_duplicates('StudyInstanceUID', keep='last')


def find_label_tables(root='/kaggle/input'):
    found = {}
    for path, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        for name in files:
            if (name in PUBLIC_LABEL_TABLES or (name.startswith(OWN_PREFIX) and name.endswith('.csv'))
                    or any(fnmatch.fnmatch(name, pattern) for pattern in OWN_JSONL_READERS)):
                found.setdefault(name, []).append(os.path.join(path, name))
    duplicates = {k: v for k, v in found.items() if len(v) > 1}
    assert not duplicates, f'Attach each label table once: {duplicates}'
    flash = [k for k in found if any(fnmatch.fnmatch(k, p) for p in OWN_JSONL_READERS)]
    assert len(flash) <= 1, f'More than one Flash member file attached: {flash}'
    return {k: v[0] for k, v in sorted(found.items())}


def read_reader(path, label_cols):
    name = os.path.basename(path)
    if name.endswith('.jsonl'):
        protocol = next(v for p, v in OWN_JSONL_READERS.items() if fnmatch.fnmatch(name, p))
        frame = states_frame(iter_jsonl(path), label_cols, protocol)
        columns = list(label_cols)
    else:
        suffix = PUBLIC_LABEL_TABLES.get(name, '')
        frame = pd.read_csv(path, dtype={'StudyInstanceUID': str})
        columns = [c + suffix for c in label_cols]
    missing = [c for c in columns if c not in frame.columns]
    assert not missing, f'{name} lacks {missing[:3]}'
    assert not frame['StudyInstanceUID'].duplicated().any(), f'{name} repeats studies'
    values = frame.set_index('StudyInstanceUID')[columns].apply(pd.to_numeric, errors='coerce')
    values.columns = label_cols
    bad = (values < 0) | (values > 1)
    if bad.any().any():
        print(f'{name}: {int(bad.sum().sum())} values outside 0..1 treated as unobserved')
        values = values.mask(bad)
    return values


# Noisy-student targets (2026-10-08). teacher_targets.csv holds out-of-fold predictions of our own
# image models (scripts/teacher_targets.py); image models average out report noise, so their OOF
# predictions are cleaner targets than any single report reader. Gold rows never take teacher values.
TEACHER_FILE = 'teacher_targets.csv'


def find_teacher(root='/kaggle/input'):
    hits = []
    for path, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        if TEACHER_FILE in files:
            hits.append(os.path.join(path, TEACHER_FILE))
    assert len(hits) <= 1, f'Attach one teacher table, found {hits}'
    return hits[0] if hits else None


# Family -> weight for teachers. Readers only: on the public LB the slot and RadImageNet families
# lowered the blend (17 models at equal weights 0.917 < two reader folds 0.923). Students and
# third-party checkpoints are never teachers.
TEACHER_FAMILIES = {'reader_convnext': 1.0, 'reader_caformer': 1.0}


def build_teacher(root, label_cols, gold_ids=(), families=None, quantile_map=()):
    """Teacher probabilities from the out-of-fold predictions (oof_fold_N.csv) of our own models
    found under root: seeds of a family are averaged, then families by weight. Every value is
    out-of-fold. Families in quantile_map (an outside model trained on other label definitions)
    keep their per-finding ranking but take the value distribution of the weighted mean of the
    other families before the average. Returns (frame with StudyInstanceUID, label_cols, n_members; meta)."""
    families = TEACHER_FAMILIES if families is None else families
    family_of = lambda folder: re.sub(r'_s\d+$', '', folder)
    members = {}
    for path, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ('train_series', 'test_series')]
        folder = os.path.basename(path)
        oofs = sorted(f for f in files if re.fullmatch(r'oof_fold_\d+\.csv', f))
        if oofs and family_of(folder) in families:
            members.setdefault(folder, []).extend(
                pd.read_csv(os.path.join(path, f), dtype={'StudyInstanceUID': str}) for f in oofs)
    assert members, f'no teacher OOF predictions under {root}'
    gold_ids, per_family = set(gold_ids), {}
    meta = dict(members={}, families={}, weights=dict(families))
    for folder, parts in sorted(members.items()):
        frame = pd.concat(parts, ignore_index=True)
        assert frame['StudyInstanceUID'].is_unique, f'{folder}: a study is out-of-fold twice'
        assert not gold_ids & set(frame['StudyInstanceUID']), f'{folder}: gold study in OOF'
        meta['members'][folder] = dict(studies=len(frame), folds=sorted(int(f) for f in frame['fold'].unique()))
        values = frame.set_index('StudyInstanceUID')[label_cols].astype(float)
        assert ((values >= 0) & (values <= 1)).all().all(), f'{folder}: OOF outside 0..1'
        per_family.setdefault(family_of(folder), []).append(values)
    ids = sorted(set().union(*[f.index for frames in per_family.values() for f in frames]))
    num = np.zeros((len(ids), len(label_cols))); den = np.zeros_like(num)
    n_members = np.zeros(len(ids), int)
    means = {}
    for family, frames in per_family.items():
        stack = np.stack([f.reindex(ids).to_numpy(float) for f in frames])
        counts = np.isfinite(stack).sum(axis=0)
        means[family] = (np.where(counts > 0, np.nansum(stack, axis=0) / np.maximum(counts, 1), np.nan), stack)
    mapped = [f for f in quantile_map if f in means]
    if mapped:
        ref_num = sum(np.nan_to_num(m) * families[f] for f, (m, _) in means.items() if f not in quantile_map)
        ref_den = sum(np.isfinite(m) * families[f] for f, (m, _) in means.items() if f not in quantile_map)
        assert np.max(ref_den) > 0, 'quantile_map needs at least one other teacher family'
        ref = np.where(ref_den > 0, ref_num / np.maximum(ref_den, 1e-9), np.nan)
        for family in mapped:
            m = means[family][0].copy()
            for j in range(m.shape[1]):
                ok, r = np.isfinite(m[:, j]), np.isfinite(ref[:, j])
                if ok.any() and r.any():
                    pct = (pd.Series(m[ok, j]).rank(method='average').to_numpy() - .5) / ok.sum()
                    m[ok, j] = np.quantile(ref[r, j], pct)
            means[family] = (m, means[family][1])
        meta['quantile_mapped'] = mapped
    for family, (mean, stack) in means.items():
        ok = np.isfinite(mean)
        num += np.where(ok, mean, 0) * families[family]
        den += ok * families[family]
        n_members += np.isfinite(stack).all(axis=2).sum(axis=0)
        meta['families'][family] = dict(members=len(frames), studies=int(ok.all(axis=1).sum()))
    teacher = pd.DataFrame(np.where(den > 0, num / np.maximum(den, 1e-9), np.nan), columns=label_cols)
    teacher.insert(0, 'StudyInstanceUID', ids)
    teacher['n_members'] = n_members
    return teacher, meta


def apply_teacher(frame, teacher_path, label_cols, weight):
    """Mix teacher probabilities into the pseudo rows: y = (1 - a) * report + a * teacher.
    weight is a constant a, or a gate (lo, hi): a = lo + (hi - lo) * u with u = 1 - |2 * report - 1|,
    so cells where the report readers agree confidently (consensus near 0 or 1) keep more of the
    report and split or uncertain cells lean on the image teacher. A pseudo cell without a report
    label takes the teacher value at cell weight hi. Gold rows are returned unchanged.
    Returns (frame, summary)."""
    lo, hi = (weight, weight) if np.isscalar(weight) else tuple(weight)
    assert 0 <= lo <= hi <= 1 and hi > 0
    teacher = pd.read_csv(teacher_path, dtype={'StudyInstanceUID': str})
    assert teacher['StudyInstanceUID'].is_unique, 'teacher repeats studies'
    t = teacher.set_index('StudyInstanceUID')[label_cols].apply(pd.to_numeric, errors='coerce')
    assert (t.isna() | ((t >= 0) & (t <= 1))).all().all(), 'teacher values outside 0..1'
    out = frame.copy()
    rows = np.flatnonzero(out['source'].eq('pseudo').to_numpy())
    wcols = [f'w_{c}' for c in label_cols]
    y = np.array(out.loc[out.index[rows], label_cols], dtype=float)
    w = np.array(out.loc[out.index[rows], wcols], dtype=float)
    tv = np.array(t.reindex(out['StudyInstanceUID'].iloc[rows]), dtype=float)
    has_t = np.isfinite(tv)
    both, only_t = has_t & np.isfinite(y), has_t & ~np.isfinite(y)
    a = lo + (hi - lo) * (1 - np.abs(2 * np.where(np.isfinite(y), y, .5) - 1))
    y[both] = (1 - a[both]) * y[both] + a[both] * tv[both]
    y[only_t] = tv[only_t]
    w[only_t] = hi
    out.loc[out.index[rows], label_cols] = np.where(np.isfinite(y), np.clip(y, .01, .99), np.nan).astype(np.float32)
    out.loc[out.index[rows], wcols] = w.astype(np.float32)
    summary = dict(teacher=teacher_path, weight=[lo, hi], mean_teacher_share=float(a[both].mean()) if both.any() else 0.,
                   studies=int(has_t.any(axis=1).sum()),
                   pseudo_studies=len(rows), mixed_cells=int(both.sum()), filled_cells=int(only_t.sum()),
                   sha256=hashlib.sha256(open(teacher_path, 'rb').read()).hexdigest())
    return out, summary


def report_groups(reports):
    # Byte-identical (up to case/punctuation) reports share targets: keep them in one fold.
    norm = reports.fillna('').astype(str).str.lower().map(lambda t: re.sub(r'\W+', '', t))
    return norm.map(lambda t: hashlib.sha1(t.encode()).hexdigest()[:16]).to_numpy()


def build_targets(train, label_cols, tables, reader_weights=None, gold_weight=3.0):
    """train: competition train.csv. Returns (frame, report) where frame holds soft
    targets, per-cell weights (w_<col>), source and report group per study."""
    ids = train['StudyInstanceUID'].astype(str)
    readers = {name: read_reader(path, label_cols).reindex(ids) for name, path in tables.items()}
    assert readers, 'Attach at least one report-label table'
    weights = {name: (reader_weights or {}).get(name, 1.0) for name in readers}
    num = sum(r[label_cols].fillna(0).to_numpy() * weights[n] for n, r in readers.items())
    den = sum(r[label_cols].notna().to_numpy() * weights[n] for n, r in readers.items())
    with np.errstate(invalid='ignore', divide='ignore'):
        soft = np.where(den > 0, num / np.maximum(den, 1e-9), np.nan)
    soft = np.clip(soft, .01, .99)
    gold = train[label_cols].apply(pd.to_numeric, errors='coerce')
    is_gold = gold.notna().all(axis=1).to_numpy()
    y = soft.astype(np.float32)
    w = np.where(np.isfinite(soft), 1.0, 0.0).astype(np.float32)
    y[is_gold] = gold.to_numpy(np.float32)[is_gold]
    w[is_gold] = gold_weight
    frame = pd.DataFrame(y, columns=label_cols)
    frame.insert(0, 'StudyInstanceUID', ids.to_numpy())
    for k, c in enumerate(label_cols): frame[f'w_{c}'] = w[:, k]
    frame['source'] = np.where(is_gold, 'gold', 'pseudo')
    frame['report_group'] = report_groups(train['Report']) if 'Report' in train else ids.to_numpy()
    # Diagnostic only: each reader against the image-read gold labels. Not used for any choice.
    report = {}
    for name, r in {**readers, 'consensus': pd.DataFrame(soft, index=ids, columns=label_cols)}.items():
        aucs = []
        # A reader that never labelled the gold studies has no gold AUC (not 0.5).
        gold_rows = int(r.loc[ids[is_gold], label_cols].notna().any(axis=1).sum())
        for c in label_cols:
            truth, score = gold.loc[is_gold, c].to_numpy(), r.loc[ids[is_gold], c].fillna(.5).to_numpy()
            aucs.append(roc_auc_score(truth, score) if len(np.unique(truth)) == 2 and gold_rows else np.nan)
        finite = [a for a in aucs if np.isfinite(a)]
        report[name] = dict(gold_macro_auc=float(np.mean(finite)) if finite else float('nan'),
                            gold_rows=gold_rows, weight=weights.get(name, 0),
                            coverage=float(np.isfinite(r.to_numpy()).mean()))
    return frame, report

# ---- cache shards -------------------------------------------------------------
# Rows are addressed as (shard, row in shard); every shard shares one recipe.
parts = []
for shard_id, directory in enumerate(CACHE_DIRS):
    part = pd.read_csv(os.path.join(directory, "cache_index.csv"))
    shape = np.load(os.path.join(directory, "cache.npy"), mmap_mode="r").shape
    assert shape[1:] == (len(PLANES), N_SLICES, CACHE_IMG, CACHE_IMG), f'{directory}: cache shape {shape}'
    assert len(part) == shape[0] and part['row'].between(0, shape[0] - 1).all()
    parts.append(part.assign(cache_id=shard_id))
    print(f"cache shard {shard_id}: {shape[0]:,} studies, {np.prod(shape)/1e9:.1f} GB  ({directory})")
index = pd.concat(parts, ignore_index=True)
assert index['StudyInstanceUID'].is_unique, 'a study appears in two cache shards'
T_TRAIN = min(T_TRAIN, N_SLICES)

# ---- targets: report-label readers + gold ------------------------------------
train_csv = pd.read_csv(os.path.join(COMP_DIR, 'train.csv'), dtype={'StudyInstanceUID': str})
LABEL_TABLES = find_label_tables(LABEL_ROOT)
print('label readers:', list(LABEL_TABLES))
_missing = [r for r in REQUIRED_READERS if not any(fnmatch.fnmatch(name, r) for name in LABEL_TABLES)]
assert not _missing, f'Attach the label inputs providing {_missing} (see KAGGLE_RUN/README.md)'
targets, label_report = build_targets(train_csv, LABEL_COLS, LABEL_TABLES, READER_WEIGHTS, GOLD_WEIGHT)
print(pd.DataFrame(label_report).T.round(3).to_string())
print('(gold AUC of each reader is a diagnostic only; it chooses nothing. NaN: the reader has no gold rows)')
TEACHER_SUMMARY = None
if np.max(TEACHER_WEIGHT) > 0:
    _teacher = find_teacher(LABEL_ROOT)
    if _teacher is None:
        # No teacher table attached: build it here from the attached reader session outputs
        # (their out-of-fold predictions), exactly as 07_teacher_targets would.
        _gold = set(targets.loc[targets['source'].eq('gold'), 'StudyInstanceUID'])
        _frame, _meta = build_teacher(LABEL_ROOT, LABEL_COLS, gold_ids=_gold, families=globals().get('TEACHER_FAMILY_WEIGHTS'),
                                      quantile_map=globals().get('TEACHER_QUANTILE_MAP', ()))
        print('teacher built in session from:', json.dumps(_meta['members']))
        _absent = sorted(f for f, w in (globals().get('TEACHER_FAMILY_WEIGHTS') or {}).items() if w > 0 and f not in _meta['families'])
        assert not _absent, f'Attach the teacher inputs for families {_absent}'
        os.makedirs(OUT, exist_ok=True)
        _teacher = os.path.join(OUT, 'teacher_targets_used.csv')
        write_atomic(_teacher, lambda p: _frame.to_csv(p, index=False))
    targets, TEACHER_SUMMARY = apply_teacher(targets, _teacher, LABEL_COLS, TEACHER_WEIGHT)
    print('teacher targets:', TEACHER_SUMMARY)
write_atomic(os.path.join(OUT, 'targets_used.csv'), lambda p: targets.to_csv(p, index=False))

WEIGHT_COLS = [f'w_{c}' for c in LABEL_COLS]
df = index[index["n_present"] > 0].merge(targets, on="StudyInstanceUID", how="inner")
df = df[df[LABEL_COLS].notna().any(axis=1)].reset_index(drop=True)
print(f"\nusable studies: {len(df):,}  ({df['source'].value_counts().to_dict()})")
# Report-embedding auxiliary targets (training only): NaN rows have no embedding and are masked.
REP_COLS, AUX_DIM, REPORT_AUX_SUMMARY = [], 0, None
if globals().get('REPORT_AUX', 0.0) > 0:
    _rep_path = one_input('report_embeddings.csv', root=LABEL_ROOT)
    _rep = pd.read_csv(_rep_path, dtype={'StudyInstanceUID': str})
    REP_COLS = [c for c in _rep.columns if c.startswith('rep_')]
    assert REP_COLS and _rep['StudyInstanceUID'].is_unique and not set(REP_COLS) & set(df.columns)
    df = df.merge(_rep[['StudyInstanceUID', *REP_COLS]], on='StudyInstanceUID', how='left')
    AUX_DIM = len(REP_COLS)
    import hashlib as _hashlib
    REPORT_AUX_SUMMARY = dict(weight=REPORT_AUX, dims=AUX_DIM,
                              studies=int(df[REP_COLS[0]].notna().sum()),
                              sha256=_hashlib.sha256(open(_rep_path, 'rb').read()).hexdigest())
    print('report auxiliary targets:', REPORT_AUX_SUMMARY)
y_all = df[LABEL_COLS].to_numpy(np.float32)
print(f"observed label cells: {(~np.isnan(y_all)).mean():.1%}")
print("\nmean soft target (observed cells):")
print(pd.Series(np.nanmean(y_all[df['source'].eq('pseudo')], axis=0), index=LABEL_COLS).round(3).to_string())
assert df['source'].isin(['gold','pseudo']).all(), 'Unknown label source'
assert ((np.isnan(y_all)) | ((y_all>=0)&(y_all<=1))).all(), 'Targets outside [0,1]'
gold_check = train_csv.set_index('StudyInstanceUID')[LABEL_COLS].apply(pd.to_numeric, errors='coerce')
gold_ids = df.loc[df['source'].eq('gold'), 'StudyInstanceUID']
assert np.array_equal(df.loc[df['source'].eq('gold'), LABEL_COLS].to_numpy(float),
                      gold_check.loc[gold_ids].to_numpy(float)), 'Gold rows must carry the official labels'

In [ ]:
"""Fold creation and metrics with explicit gold exclusion."""
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold


def assign_folds(df, label_cols, n_splits=5, seed=42, group_col=None):
    folds = np.full(len(df), -1, dtype=int)
    pseudo = np.flatnonzero(df['source'].eq('pseudo').to_numpy())
    assert len(pseudo) >= n_splits
    y = df.iloc[pseudo][label_cols].to_numpy(float)
    # Balance missingness as well as positive findings. No gold target enters the split.
    strata = np.concatenate([np.nan_to_num(y, nan=0) >= .5, np.isfinite(y)], axis=1).astype(int)
    groups = df.iloc[pseudo][group_col].astype(str).to_numpy() if group_col else df.iloc[pseudo]['StudyInstanceUID'].to_numpy()
    if len(np.unique(groups)) < len(groups):
        splitter = GroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = splitter.split(pseudo, groups=groups)
    else:
        splitter = MultilabelStratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        splits = splitter.split(pseudo, strata)
    for fold, (_, val) in enumerate(splits): folds[pseudo[val]] = fold
    assert (folds[pseudo] >= 0).all()
    assert (folds[df['source'].eq('gold')] == -1).all()
    return folds


def macro_auc(y, scores, label_cols):
    aucs = {}
    for k, name in enumerate(label_cols):
        ok = np.isfinite(y[:, k]) & np.isfinite(scores[:, k])
        binary = (y[ok, k] >= .5).astype(int)
        aucs[name] = roc_auc_score(binary, scores[ok, k]) if len(np.unique(binary)) == 2 else np.nan
    valid = [v for v in aucs.values() if np.isfinite(v)]
    return (float(np.mean(valid)) if valid else float('nan')), aucs


def rank_blend(predictions):
    arrays = [np.asarray(p, dtype=float) for p in predictions]
    assert arrays and all(a.shape == arrays[0].shape for a in arrays)
    assert all(np.isfinite(a).all() for a in arrays)
    return np.mean([pd.DataFrame(a).rank(pct=True, method='average').to_numpy() for a in arrays], axis=0)

In [ ]:
"""Training dataset, embedded in training.ipynb after the fold helpers."""
import cv2

class KneeDS(Dataset):
    def __init__(self, df, rows, train):
        self.df = df.reset_index(drop=True)
        self.rows = rows
        self.train = train
        self.y = self.df[LABEL_COLS].to_numpy(np.float32)
        # Per-cell loss weights (gold rows are up-weighted); unweighted tables default to 1.
        weight_cols = [f'w_{c}' for c in LABEL_COLS]
        self.w = (self.df[weight_cols].to_numpy(np.float32) if set(weight_cols) <= set(self.df)
                  else np.ones_like(self.y))
        self.present = self.df[[f"present_{s}" for s in range(len(PLANES))]].to_numpy(bool)
        self.row_of = self.df["row"].to_numpy(int)
        self.shard_of = (self.df["cache_id"].to_numpy(int) if "cache_id" in self.df
                         else np.zeros(len(self.df), int))
        self.caches = {}                       # memmaps opened lazily, per worker and shard
        # Training batches carry the report-embedding targets after the labels (weight 1 where known).
        rep = globals().get('REP_COLS') or []
        self.aux = self.df[rep].to_numpy(np.float32) if (train and rep) else None

    def _cache(self, shard):
        if shard not in self.caches:
            self.caches[shard] = np.load(os.path.join(CACHE_DIRS[shard], "cache.npy"), mmap_mode="r")
        return self.caches[shard]

    def __len__(self):
        return len(self.rows)

    def _aug(self, vol):
        # NO horizontal or vertical flip: a horizontal flip in the coronal plane swaps
        # medial and lateral, which would invert four of the twelve labels.
        h, w = vol.shape[-2:]
        ang = np.random.uniform(-12, 12)
        sc  = np.random.uniform(0.9, 1.1)
        M = cv2.getRotationMatrix2D((w/2, h/2), ang, sc)
        M[:, 2] += np.random.uniform(-0.06, 0.06, 2) * w
        out = np.empty_like(vol)
        for i in range(vol.shape[0]):
            out[i] = cv2.warpAffine(vol[i], M, (w, h), flags=cv2.INTER_LINEAR,
                                    borderValue=0)
        if np.random.rand() < 0.5:
            out = np.clip(out.astype(np.float32) * np.random.uniform(0.85, 1.15)
                          + np.random.uniform(-15, 15), 0, 255).astype(np.uint8)
        return out

    def _aug_slot(self, vol):
        # Public-member jitter for slot_query: small rotation, zoom-in only (zooming out
        # would invent tissue at the crop edge, where the popliteal fossa sits), edge
        # replication, multiplicative intensity. Still no flips.
        h, w = vol.shape[-2:]
        M = cv2.getRotationMatrix2D((w/2, h/2), np.random.uniform(-8, 8),
                                    np.random.uniform(1.0, 1.08))
        M[:, 2] += np.random.uniform(-0.025, 0.025, 2) * w
        out = np.stack([cv2.warpAffine(s, M, (w, h), flags=cv2.INTER_LINEAR,
                                       borderMode=cv2.BORDER_REPLICATE) for s in vol])
        return np.clip(out.astype(np.float32) * np.random.uniform(0.9, 1.1),
                       0, 255).astype(np.uint8)

    def __getitem__(self, i):
        j = self.rows[i]
        r = self.row_of[j]
        cache = self._cache(self.shard_of[j])
        present = self.present[j]

        if self.train and ARCH == 'reader25d':
            # All windows; the model subsamples windows and augments each series on GPU.
            vol = np.asarray(cache[r])
        elif self.train and ARCH == 'slot_query':
            # One slice from each third of the band, drawn independently per slot, so
            # every cached position trains while each input still spans the joint.
            parts = slice_thirds(cache.shape[2])
            vol = np.stack([self._aug_slot(np.asarray(cache[r, s, [np.random.choice(p) for p in parts]]))
                            if present[s] else np.zeros((3, *cache.shape[3:]), np.uint8)
                            for s in range(cache.shape[1])])
        elif self.train:
            # Slice the memmap BEFORE materialising, so the discarded slices are
            # never read off disk at all (~37% less I/O per sample).
            avail = cache.shape[2]
            t = min(T_TRAIN, avail)
            t0 = np.random.randint(0, avail - t + 1)
            vol = np.asarray(cache[r][:, t0:t0 + t])
            vol = np.stack([self._aug(v) if present[s] else v
                            for s, v in enumerate(vol)])
        else:
            vol = np.asarray(cache[r])               # eval uses every slice

        y, w = self.y[j], self.w[j]
        if self.aux is not None:
            y = np.concatenate([y, self.aux[j]])
            w = np.concatenate([w, np.isfinite(self.aux[j]).astype(np.float32)])
        return (torch.from_numpy(np.array(vol, copy=True, order='C')),
                torch.from_numpy(present.copy()),
                torch.from_numpy(y.copy()),
                torch.from_numpy(w.copy()))


def collate(batch):
    xs, ps, ys, ws = zip(*batch)
    return torch.stack(xs), torch.stack(ps), torch.stack(ys), torch.stack(ws)

## 2 · Model

`slot_query` (default) follows the public DINOv2 members: each series slot is one 3-channel image whose channels are slices from the lower, middle and upper thirds of the stack. A DINOv2-S encoder has its last six blocks fine-tuned at 8e-6 while the head learns at 1e-3. CLS and mean patch tokens feed a head where every finding has its own query attending over slots. Evaluation averages logits over every spread triplet of the cached slices.

`kneenet` is the earlier EfficientNet-b0 model with neighbour triplets, slice attention and per-label plane attention, kept for comparison and for older checkpoints.

In [ ]:
"""Shared model architecture for training and offline inference."""
import numpy as np
import torch
import torch.nn as nn
import timm

def make_neighbours(x):
    # (B,S,T,H,W) -> (B,S,T,3,H,W) from slices t-1,t,t+1 (edges clamped)
    prev = torch.cat([x[:, :, :1], x[:, :, :-1]], dim=2)
    nxt  = torch.cat([x[:, :, 1:], x[:, :, -1:]], dim=2)
    return torch.stack([prev, x, nxt], dim=3)


class AttnPool(nn.Module):
    # Attention over slices within one plane.
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def forward(self, x, mask=None):               # x (B,T,D)
        a = self.score(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(~mask, torch.finfo(a.dtype).min)
        a = a.softmax(dim=-1)
        return (x * a.unsqueeze(-1)).sum(1), a


class LabelAttnPool(nn.Module):
    # Per-label attention over planes: each finding picks the view it needs.
    def __init__(self, dim, n_labels):
        super().__init__()
        self.score = nn.Linear(dim, n_labels)

    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)
        a = self.score(x)                          # (B,S,L)
        a = a.masked_fill(~mask.unsqueeze(-1), torch.finfo(a.dtype).min)
        a = a.softmax(dim=1)                       # over planes, per label
        z = torch.einsum("bsd,bsl->bld", x, a)     # (B,L,D)
        return z, a


class KneeNet(nn.Module):
    def __init__(self, backbone="tf_efficientnet_b0.ns_jft_in1k", n_out=12, n_planes=3,
                 pretrained=True, drop=0.3, img=224, normalization="legacy",
                 encoder_chunk=64):
        super().__init__()
        kwargs = {"img_size": img} if backbone.startswith("vit_") else {}
        self.enc = timm.create_model(backbone, pretrained=pretrained,
                                     num_classes=0, in_chans=3, **kwargs)
        self.encoder_chunk = encoder_chunk
        config = timm.data.resolve_model_data_config(self.enc)
        mean = config['mean'] if normalization == 'pretrained' else (0.449,)*3
        std = config['std'] if normalization == 'pretrained' else (0.226,)*3
        self.register_buffer('input_mean', torch.tensor(mean).view(1,3,1,1), persistent=False)
        self.register_buffer('input_std', torch.tensor(std).view(1,3,1,1), persistent=False)
        d = self.enc.num_features
        self.slice_pool = AttnPool(d)
        self.plane_emb = nn.Parameter(torch.zeros(n_planes, d))
        self.plane_pool = LabelAttnPool(d, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))
        nn.init.trunc_normal_(self.head_w, std=0.02)

    def forward(self, x, present):
        # autocast entered HERE, not in the training loop: it is thread-local, so a
        # loop-level autocast never reaches a threaded forward. Verified reaching the
        # encoder as float16.
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            B, S, T, H, W = x.shape
            x = x.float().div(255.)   # ImageNet grey stats
            x3 = make_neighbours(x).reshape(B * S * T, 3, H, W)

            x3 = (x3 - self.input_mean) / self.input_std
            # Chunk evaluation to bound memory; training keeps BatchNorm batch statistics intact.
            if not self.training and self.encoder_chunk and len(x3) > self.encoder_chunk:
                features = torch.cat([self.enc(part) for part in x3.split(self.encoder_chunk)])
            else:
                features = self.enc(x3)
            f = features.reshape(B * S, T, -1)
            v, _ = self.slice_pool(f)
            v = v.reshape(B, S, -1) + self.plane_emb.unsqueeze(0)

            m = present.clone()
            m[~m.any(dim=1)] = True                # guard: never all-masked
            z, plane_att = self.plane_pool(v, m)
            z = self.drop(z)
            logits = (z * self.head_w.unsqueeze(0)).sum(-1) + self.head_b
        return logits, plane_att



def slice_thirds(n):
    # Lower, middle and upper thirds of the cached slice band.
    parts = np.array_split(np.arange(n), 3)
    if min(len(p) for p in parts) == 0:
        return [np.array([0]), np.array([(n - 1) // 2]), np.array([n - 1])]
    return parts


def spread_groups(n):
    # 3-slice inputs that each span the band: one slice from every third. For six cached
    # slices that is [0, 2, 4] and [1, 3, 5]; a 3-slice input is its own single group.
    parts = slice_thirds(n)
    return [[int(p[g]) for p in parts] for g in range(min(len(p) for p in parts))]


class SlotQueryHead(nn.Module):
    # Per-finding query attention over series slots (public DINOv2 member head).
    def __init__(self, dim, n_slots, n_out, hidden=256, drop=0.2):
        super().__init__()
        self.proj = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, hidden), nn.GELU())
        self.slot_emb = nn.Parameter(torch.randn(n_slots, hidden) * 0.02)
        self.query = nn.Parameter(torch.randn(n_out, hidden) * 0.02)
        self.drop = nn.Dropout(drop)
        self.out = nn.Linear(hidden, n_out)
        self.hidden = hidden

    def forward(self, x, mask):                    # x (B,S,D), mask (B,S)
        h = self.proj(x) + self.slot_emb
        a = torch.einsum("bsh,oh->bos", h, self.query) / self.hidden ** 0.5
        a = a.masked_fill(~mask.unsqueeze(1), torch.finfo(a.dtype).min).softmax(-1)
        z = self.drop(torch.einsum("bos,bsh->boh", a, h))
        return (z * self.out.weight.unsqueeze(0)).sum(-1) + self.out.bias, a


class SlotQueryNet(nn.Module):
    """One 3-slice image per series slot, encoder partly fine-tuned, query head on top.

    Training feeds one spread triplet per slot (T=3). Evaluation feeds every cached slice
    and averages logits over the spread groups, so no slice position is left unseen.
    """
    def __init__(self, backbone="vit_small_patch14_dinov2.lvd142m", n_out=12, n_slots=6,
                 pretrained=True, img=336, drop=0.2, hidden=256, unfreeze_last=6,
                 normalization="pretrained", encoder_chunk=48):
        super().__init__()
        kwargs = {"img_size": img} if backbone.startswith("vit_") else {}
        self.enc = timm.create_model(backbone, pretrained=pretrained,
                                     num_classes=0, in_chans=3, **kwargs)
        self.img = img
        self.encoder_chunk = encoder_chunk
        self.tokens = hasattr(self.enc, "blocks") and hasattr(self.enc, "num_prefix_tokens")
        config = timm.data.resolve_model_data_config(self.enc)
        mean = config['mean'] if normalization == 'pretrained' else (0.449,)*3
        std = config['std'] if normalization == 'pretrained' else (0.226,)*3
        self.register_buffer('input_mean', torch.tensor(mean).view(1,3,1,1), persistent=False)
        self.register_buffer('input_std', torch.tensor(std).view(1,3,1,1), persistent=False)
        if unfreeze_last is not None and hasattr(self.enc, "blocks"):
            # Early blocks stay at their self-supervised weights; only late blocks adapt.
            for p in self.enc.parameters(): p.requires_grad = False
            blocks = list(self.enc.blocks)
            for block in blocks[max(0, len(blocks) - unfreeze_last):]:
                for p in block.parameters(): p.requires_grad = True
            for name in ("norm", "fc_norm"):
                module = getattr(self.enc, name, None)
                if isinstance(module, nn.Module):
                    for p in module.parameters(): p.requires_grad = True
        dim = self.enc.num_features * 2
        self.head = SlotQueryHead(dim, n_slots, n_out, hidden=hidden, drop=drop)

    def encode(self, x):                           # (N,3,H,W) normalised -> (N,2D)
        f = self.enc.forward_features(x)
        if self.tokens:
            # CLS token plus mean of patch tokens; register tokens are excluded.
            return torch.cat([f[:, 0], f[:, self.enc.num_prefix_tokens:].mean(1)], dim=1)
        if f.ndim == 4:
            return torch.cat([f.mean((-2, -1)), f.amax((-2, -1))], dim=1)
        return torch.cat([f, f], dim=1)

    def forward(self, x, present):
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            B, S, T, H, W = x.shape
            m = present.clone()
            m[~m.any(dim=1)] = True                # guard: never all-masked
            keep = m.reshape(-1)
            groups = spread_groups(T)
            logits, attention = 0., 0.
            for group in groups:
                images = x[:, :, group].reshape(B * S, 3, H, W)[keep].float().div(255.)
                if H != self.img:
                    images = nn.functional.interpolate(images, size=(self.img, self.img),
                                                       mode="bilinear", align_corners=False)
                images = (images - self.input_mean) / self.input_std
                # Absent slots are never encoded; their features stay zero and masked.
                if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:
                    f = torch.cat([self.encode(part) for part in images.split(self.encoder_chunk)])
                else:
                    f = self.encode(images)
                features = f.new_zeros(B * S, f.shape[-1])
                features[keep] = f
                z, a = self.head(features.reshape(B, S, -1), m)
                logits, attention = logits + z, attention + a
        return logits / len(groups), attention / len(groups)


class Reader25D(nn.Module):
    """2.5D study reader (public ConvNeXt reader layout).

    Every cached window of three adjacent slices is one CNN image. Its feature becomes a
    token tagged with its series slot and its position along the stack; a small
    transformer mixes all tokens of the study and each finding pools them with its own
    attention. Training augments each series with one shared rigid/intensity jitter (no
    flips: laterality is already normalised) and encodes a random subset of windows.
    """
    def __init__(self, backbone="convnext_tiny.in12k_ft_in1k", n_out=12, n_slots=6,
                 pretrained=True, img=240, dim=384, depth=2, drop=0.2, drop_path=0.1,
                 normalization="pretrained", band=(0.06, 0.94), train_windows=6,
                 encoder_chunk=96, slot_dropout=0.0, aux_dim=0):
        super().__init__()
        self.slot_dropout = slot_dropout     # training only: hide whole series (protocols vary by site)
        # Runtime switches for XLA/TPU training; neither changes the parameters or the checkpoint.
        # static_shapes: encode a fixed number of windows for every slot (absent ones masked), so
        # tensor shapes never depend on the data. augment_in_model=False: the data loader already
        # applied the per-series jitter (augment_series_cpu), so the model only normalises.
        self.static_shapes = False
        self.augment_in_model = True
        # Test-time window subsets (eval only; inference WINDOW_TTA): encode a random subset of this many
        # windows per series, as in training, instead of all of them. None: every window.
        self.subset_windows = None
        kwargs = {"drop_path_rate": drop_path} if drop_path else {}
        if backbone.startswith("vit_"):
            kwargs["img_size"] = img
        self.enc = timm.create_model(backbone, pretrained=pretrained, num_classes=0, **kwargs)
        self.img, self.band, self.train_windows = img, tuple(band), train_windows
        self.encoder_chunk = encoder_chunk
        config = timm.data.resolve_model_data_config(self.enc)
        mean = config['mean'] if normalization == 'pretrained' else (0.449,)*3
        std = config['std'] if normalization == 'pretrained' else (0.226,)*3
        self.register_buffer('input_mean', torch.tensor(mean).view(1,3,1,1), persistent=False)
        self.register_buffer('input_std', torch.tensor(std).view(1,3,1,1), persistent=False)
        self.proj = nn.Sequential(nn.Dropout(drop), nn.Linear(self.enc.num_features, dim),
                                  nn.LayerNorm(dim))
        self.slot_emb = nn.Embedding(n_slots, dim)
        self.pos_emb = nn.Sequential(nn.Linear(1, dim), nn.GELU(), nn.Linear(dim, dim))
        layer = nn.TransformerEncoderLayer(dim, 8, dim * 2, dropout=0.1, batch_first=True,
                                           norm_first=True)
        self.tf = nn.TransformerEncoder(layer, depth, enable_nested_tensor=False) if depth else None
        self.att = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop),
                                 nn.Linear(256, n_out))
        self.cls_w = nn.Parameter(torch.randn(n_out, dim) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        # Report-grounded auxiliary head (training only): regresses the study's report embedding
        # from the finding-pooled and the mean study tokens. Its output is kept in last_aux; the
        # logits and the inference path do not depend on it.
        self.aux = (nn.Sequential(nn.Linear(2 * dim, dim), nn.GELU(), nn.Linear(dim, aux_dim))
                    if aux_dim else None)
        self.last_aux = None

    def _aux(self, h, pad, pooled):
        if self.aux is None:
            return
        keep = (~pad).unsqueeze(-1).to(h.dtype)
        mean = (h * keep).sum(1) / keep.sum(1).clamp_min(1)
        self.last_aux = self.aux(torch.cat([pooled.mean(1), mean], -1))

    def views(self, x, series, n_series):
        # x float [N,3,H,W] in 0..1; one random transform per series in training.
        if x.shape[-1] != self.img:
            x = nn.functional.interpolate(x, size=(self.img, self.img), mode="bilinear",
                                          align_corners=False, antialias=x.shape[-1] > self.img)
        if self.training and self.augment_in_model:
            dev = x.device
            ang = (torch.rand(n_series, device=dev) - .5) * 2 * np.radians(12)
            sc = 1 + (torch.rand(n_series, device=dev) - .5) * .24
            tx, ty = ((torch.rand(2, n_series, device=dev) - .5) * .12)
            theta = torch.stack([torch.stack([torch.cos(ang) * sc, -torch.sin(ang) * sc, tx], 1),
                                 torch.stack([torch.sin(ang) * sc, torch.cos(ang) * sc, ty], 1)], 1)
            grid = nn.functional.affine_grid(theta[series], x.shape, align_corners=False)
            x = nn.functional.grid_sample(x, grid, mode="bilinear", padding_mode="zeros",
                                          align_corners=False)
            gamma = torch.exp((torch.rand(n_series, device=dev) - .5) * .5)[series]
            gain = (1 + (torch.rand(n_series, device=dev) - .5) * .2)[series]
            x = x.clamp_min(1e-4).pow(gamma.view(-1, 1, 1, 1)) * gain.view(-1, 1, 1, 1)
        return (x - self.input_mean) / self.input_std

    def forward(self, x, present):
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            B, S, T, H, W = x.shape
            K = T // 3
            m = present.clone()
            if self.training and self.slot_dropout > 0:
                kept = m & (torch.rand(m.shape, device=m.device) >= self.slot_dropout)
                # A study that lost every series keeps one of its present series, chosen at random.
                pick = torch.rand(m.shape, device=m.device).masked_fill(~m, -1).argmax(dim=1)
                lost = ~kept.any(dim=1) & m.any(dim=1)
                one = nn.functional.one_hot(pick, m.shape[1]).bool()
                m = kept | (lost.unsqueeze(1) & one)   # fixed-shape boolean algebra (XLA-safe)
            m = m | ~m.any(dim=1, keepdim=True)    # guard: never all-masked
            if self.static_shapes:
                return self._forward_static(x, m, B, S, K, H, W)
            tokens = m.unsqueeze(-1).expand(B, S, K).clone()
            n_win = self.train_windows if self.training else getattr(self, 'subset_windows', None)
            if n_win and n_win < K:
                # Encode a random subset of windows per series: stack augmentation that
                # also bounds activation memory. Positions stay exact.
                rank = torch.rand(B, S, K, device=x.device).argsort(-1).argsort(-1)
                tokens &= rank < n_win
            keep = tokens.reshape(-1)
            series = torch.arange(B * S, device=x.device).repeat_interleave(K)[keep]
            images = x[:, :, :3 * K].reshape(B * S * K, 3, H, W)[keep].float().div(255.)
            images = self.views(images, series, B * S).contiguous(memory_format=torch.channels_last)
            if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:
                f = torch.cat([self.enc(part) for part in images.split(self.encoder_chunk)])
            else:
                f = self.enc(images)
            feat = f.new_zeros(B * S * K, f.shape[-1])
            feat[keep] = f
            slot = torch.arange(S, device=x.device).view(1, S, 1).expand(B, S, K)
            pos = torch.linspace(self.band[0], self.band[1], K, device=x.device)
            h = (self.proj(feat).view(B, S, K, -1) + self.slot_emb(slot)
                 + self.pos_emb(pos.view(1, 1, K, 1).to(feat.dtype)))
            h = h.reshape(B, S * K, -1)
            pad = ~tokens.reshape(B, S * K)
            if self.tf is not None:
                h = self.tf(h, src_key_padding_mask=pad)
            a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)
            pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)
            logits = (pooled * self.cls_w).sum(-1) + self.cls_b
            self._aux(h, pad, pooled)
        return logits, a


def _reader_forward_static(self, x, m, B, S, K, H, W):
    """Reader25D forward with data-independent shapes (XLA/TPU). Every slot encodes the same
    number of windows; absent or dropped slots are encoded too but masked, exactly as the
    dynamic path masks them, so outputs agree up to floating-point error."""
    n_win = self.train_windows if self.training else getattr(self, 'subset_windows', None)
    n_win = n_win if n_win and n_win < K else K
    if n_win < K:
        idx = torch.rand(B, S, K, device=x.device).argsort(-1)[..., :n_win].sort(-1).values
    else:
        idx = torch.arange(K, device=x.device).view(1, 1, K).expand(B, S, K)
    windows = x[:, :, :3 * K].reshape(B, S, K, 3, H, W)
    windows = torch.gather(windows, 2, idx.reshape(B, S, n_win, 1, 1, 1).expand(B, S, n_win, 3, H, W))
    images = windows.reshape(B * S * n_win, 3, H, W).float().div(255.)
    series = torch.arange(B * S, device=x.device).repeat_interleave(n_win)
    images = self.views(images, series, B * S)
    if not self.training and self.encoder_chunk and len(images) > self.encoder_chunk:
        f = torch.cat([self.enc(part) for part in images.split(self.encoder_chunk)])
    else:
        f = self.enc(images)
    tokens = m.unsqueeze(-1).expand(B, S, n_win)
    slot = torch.arange(S, device=x.device).view(1, S, 1).expand(B, S, n_win)
    pos = torch.linspace(self.band[0], self.band[1], K, device=x.device)[idx]
    h = (self.proj(f).view(B, S, n_win, -1) + self.slot_emb(slot)
         + self.pos_emb(pos.unsqueeze(-1).to(f.dtype)))
    h = h.reshape(B, S * n_win, -1)
    pad = ~tokens.reshape(B, S * n_win)
    if self.tf is not None:
        h = self.tf(h, src_key_padding_mask=pad)
    a = self.att(h).float().masked_fill(pad.unsqueeze(-1), -1e4).softmax(1)
    pooled = torch.einsum("btc,btd->bcd", a.to(h.dtype), h)
    logits = (pooled * self.cls_w).sum(-1) + self.cls_b
    self._aux(h, pad, pooled)
    return logits, a


Reader25D._forward_static = _reader_forward_static


def augment_series_cpu(x, generator=None):
    """Reader25D's training jitter on the CPU, for loaders that feed XLA devices: one rotation
    (+-12 deg), scale (+-12%), shift (+-6%), gamma and gain per series, shared by all its slices.
    x: uint8 [S, T, H, W] of one study; returns uint8 of the same shape (values clipped to 0..255)."""
    S, T, H, W = x.shape
    rnd = lambda *shape: torch.rand(*shape, generator=generator)
    ang = (rnd(S) - .5) * 2 * np.radians(12)
    sc = 1 + (rnd(S) - .5) * .24
    tx, ty = (rnd(2, S) - .5) * .12
    theta = torch.stack([torch.stack([torch.cos(ang) * sc, -torch.sin(ang) * sc, tx], 1),
                         torch.stack([torch.sin(ang) * sc, torch.cos(ang) * sc, ty], 1)], 1)
    xf = x.float().div(255.)
    grid = nn.functional.affine_grid(theta, (S, T, H, W), align_corners=False)
    xf = nn.functional.grid_sample(xf, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    gamma = torch.exp((rnd(S) - .5) * .5).view(S, 1, 1, 1)
    gain = (1 + (rnd(S) - .5) * .2).view(S, 1, 1, 1)
    xf = xf.clamp_min(1e-4).pow(gamma) * gain
    return (xf.clamp(0, 1) * 255).round().to(torch.uint8)


def window_centres(n_slices):
    # Centre slice of each cached 3-slice window (v3 layout); plain slices otherwise.
    return list(range(1, n_slices, 3)) if n_slices % 3 == 0 and n_slices >= 6 else list(range(n_slices))


class RadImageNetHead(nn.Module):
    """Frozen official RadImageNet ResNet-50 slice encoder + per-finding query head.

    Follows the public RadImageNet arm (E9/E11): every window-centre slice is one
    grey image scaled to [-1, 1]; its 2048-d global-average feature is a token tagged
    with slot and stack position; one query per finding cross-attends over all tokens
    and is fused with the mean token. Only the head trains.
    """
    def __init__(self, n_out=12, n_slots=6, n_windows=8, img=224, head_dim=512,
                 weights_path=None, encoder_chunk=128):
        super().__init__()
        from torchvision.models import resnet50
        self.backbone = nn.Sequential(*list(resnet50(weights=None).children())[:-2])
        if weights_path:
            state = torch.load(weights_path, map_location="cpu", weights_only=True)
            self.backbone.load_state_dict({k.split("backbone.", 1)[1]: v for k, v in state.items()},
                                          strict=True)
        for p in self.backbone.parameters(): p.requires_grad = False
        self.img, self.n_windows, self.encoder_chunk = img, n_windows, encoder_chunk
        dim = 2048
        self.project = nn.Sequential(nn.LayerNorm(dim), nn.Linear(dim, head_dim), nn.GELU())
        self.plane = nn.Parameter(torch.randn(n_slots, head_dim) * .01)
        self.position = nn.Parameter(torch.randn(n_windows, head_dim) * .01)
        self.query = nn.Parameter(torch.randn(n_out, head_dim) * .02)
        self.attn = nn.MultiheadAttention(head_dim, 8, dropout=.10, batch_first=True)
        self.fuse = nn.Sequential(nn.LayerNorm(head_dim * 4), nn.Linear(head_dim * 4, head_dim),
                                  nn.GELU(), nn.Dropout(.15))
        self.weight = nn.Parameter(torch.randn(n_out, head_dim) * .02)
        self.bias = nn.Parameter(torch.zeros(n_out))

    def train(self, mode=True):
        super().train(mode)
        self.backbone.eval()                       # frozen BatchNorm statistics
        return self

    @torch.no_grad()
    def encode(self, x, present):
        # x uint8 (B,S,T,H,W) -> tokens (B, S*K, 2048) float16, token mask (B, S*K)
        B, S, T, H, W = x.shape
        centres = window_centres(T)[:self.n_windows]
        K = len(centres)
        keep = present.unsqueeze(-1).expand(B, S, K).reshape(-1)
        images = x[:, :, centres].reshape(B * S * K, 1, H, W)[keep].float().div(127.5).sub(1.)
        if H != self.img:
            images = nn.functional.interpolate(images, size=(self.img, self.img), mode="bilinear",
                                               align_corners=False)
        images = images.expand(-1, 3, -1, -1)
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            f = torch.cat([self.backbone(part).mean(dim=(2, 3)) for part in images.split(self.encoder_chunk)]) \
                if len(images) else images.new_zeros(0, 2048)
        tokens = torch.zeros(B * S * K, 2048, device=x.device, dtype=torch.float16)
        tokens[keep] = f.to(torch.float16)
        return tokens.view(B, S * K, 2048), keep.view(B, S * K).float()

    def head(self, feature, mask):
        n_slots = self.plane.shape[0]
        token = self.project(feature.float())
        token = token.view(len(token), n_slots, -1, token.shape[-1])
        token = token + self.plane[None, :, None] + self.position[None, None, :token.shape[2]]
        token = token.flatten(1, 2)
        pad = mask <= 0
        empty = pad.all(1)
        if empty.any():
            pad = pad.clone(); pad[empty, 0] = False
        query = self.query.unsqueeze(0).expand(len(token), -1, -1)
        attended = query + self.attn(query, token, token, key_padding_mask=pad, need_weights=False)[0]
        mean = (token * mask.unsqueeze(-1)).sum(1, keepdim=True) / mask.sum(1, keepdim=True).clamp_min(1).unsqueeze(-1)
        mean = mean.expand(-1, len(self.query), -1)
        fused = self.fuse(torch.cat([attended, mean, (attended - mean).abs(), attended * mean], -1))
        return (fused * self.weight.unsqueeze(0)).sum(-1) + self.bias

    def forward(self, x, present):
        m = present.clone()
        m[~m.any(dim=1)] = True
        return self.head(*self.encode(x, m)), None


def build_model(cfg, n_out, n_planes, pretrained=True):
    # A checkpoint's recipe rebuilds its own architecture; old recipes have no arch key.
    arch = cfg.get('arch', 'kneenet')
    if arch == 'kneenet':
        return KneeNet(cfg['backbone'], n_out, n_planes, pretrained=pretrained,
                       drop=cfg.get('drop', .3), img=cfg['img'],
                       normalization=cfg.get('normalization', 'legacy'))
    if arch == 'slot_query':
        return SlotQueryNet(cfg['backbone'], n_out, n_planes, pretrained=pretrained,
                            img=cfg['img'], drop=cfg.get('drop', .2),
                            hidden=cfg.get('head_hidden', 256),
                            unfreeze_last=cfg.get('unfreeze_last', 6),
                            normalization=cfg.get('normalization', 'pretrained'))
    if arch == 'reader25d':
        return Reader25D(cfg['backbone'], n_out, n_planes, pretrained=pretrained,
                         img=cfg['img'], drop=cfg.get('drop', .2),
                         dim=cfg.get('head_hidden', 384), drop_path=cfg.get('drop_path', .1),
                         band=tuple(cfg['preprocessing']['slice_band']),
                         train_windows=cfg.get('train_windows', 6),
                         normalization=cfg.get('normalization', 'pretrained'),
                         slot_dropout=cfg.get('slot_dropout', 0.0), aux_dim=cfg.get('aux_dim', 0))
    if arch == 'radhead':
        # The frozen encoder ships inside every checkpoint; weights_path is training-only.
        return RadImageNetHead(n_out, n_planes, n_windows=cfg.get('n_windows', 8), img=cfg['img'],
                               head_dim=cfg.get('head_hidden', 512),
                               weights_path=cfg.get('weights_path') if pretrained else None)
    raise ValueError(f'Unknown architecture {arch!r}')

## 3 · Loss and metric

In [ ]:
def masked_bce(logits, targets, weights=None):
    # BCE over (B,12) soft targets that may contain NaN. NaN cells get exactly zero
    # gradient, so a study a reader abstained on still trains the other heads. Optional
    # per-cell weights up-weight gold rows; the mean is taken over observed weight.
    mask = ~torch.isnan(targets)
    if not mask.any():
        return logits.sum() * 0.0
    w = mask.float() if weights is None else weights * mask
    loss = F.binary_cross_entropy_with_logits(
        logits, torch.nan_to_num(targets, nan=0.0), reduction="none")
    return (loss * w).sum() / w.sum().clamp_min(1e-6)

## 4 · Train

In [ ]:
# Training, checkpoint selection, and fold exports are implemented below.

In [ ]:
"""Five-fold training runtime, embedded in training.ipynb."""
def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed); random.seed(seed)
    cv2.setNumThreads(0)


def make_loader(rows, training, seed):
    return DataLoader(KneeDS(df, rows, training), batch_size=BATCH,
                      shuffle=training, num_workers=NUM_WORKERS,
                      collate_fn=collate, pin_memory=DEV=='cuda', drop_last=False,
                      persistent_workers=NUM_WORKERS > 0, worker_init_fn=seed_worker,
                      generator=torch.Generator().manual_seed(seed),
                      **({'prefetch_factor': 2} if NUM_WORKERS else {}))


@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    ys, ps = [], []
    for x, present, y, *_ in loader:
        logits, _ = model(x.to(DEV, non_blocking=True), present.to(DEV, non_blocking=True))
        ps.append(logits.float().sigmoid().cpu().numpy()); ys.append(y.numpy())
    if not ys: return np.empty((0,len(LABEL_COLS))), np.empty((0,len(LABEL_COLS)))
    return np.concatenate(ys), np.concatenate(ps)


def atomic_torch_save(obj, path):
    temp = str(path) + '.tmp'
    torch.save(obj, temp); os.replace(temp, path)


os.makedirs(OUT, exist_ok=True)
folds = assign_folds(df, LABEL_COLS, N_FOLDS, SEED, GROUP_COL if GROUP_COL in df else None)
fold_table = df[['StudyInstanceUID', 'source']].assign(fold=folds)
write_atomic(os.path.join(OUT, 'folds.csv'), lambda p: fold_table.to_csv(p, index=False))
# Record the dataset identity so resumed folds cannot silently use changed labels or cache.
import hashlib, importlib.metadata
identity = hashlib.sha256((df.to_json(orient='split') + json.dumps(CACHE_META, sort_keys=True)
                           + fold_table.to_json()).encode()).hexdigest()
recipe = dict(arch=ARCH, backbone=BACKBONE, img=IMG, n_slices=N_SLICES, planes=PLANES,
              label_cols=LABEL_COLS, t_train=T_TRAIN, preprocessing=CACHE_META,
              unfreeze_last=UNFREEZE_LAST, head_hidden=HEAD_HIDDEN, pct_start=PCT_START,
              normalization=NORMALIZATION, drop=DROPOUT, seed=SEED,
              epochs=EPOCHS, lr=LR, weight_decay=WD, n_folds=N_FOLDS, accum_steps=ACCUM_STEPS,
              encoder_lr_scale=ENCODER_LR_SCALE, batch=BATCH, identity=identity,
              gold_weight=GOLD_WEIGHT, ema=EMA_DECAY, clip=CLIP, train_windows=TRAIN_WINDOWS, slot_dropout=SLOT_DROPOUT,
              preset=PRESET, grad_checkpoint=GRAD_CHECKPOINT, teacher=TEACHER_SUMMARY,
              group_col=GROUP_COL if GROUP_COL in df else None,
              aux_dim=globals().get('AUX_DIM', 0), report_aux=globals().get('REPORT_AUX_SUMMARY'),
              versions={p: importlib.metadata.version(p) for p in ('torch','timm','numpy')})
write_atomic(os.path.join(OUT, 'run_config.json'), json.dumps(recipe, indent=2))
REPORT_AUX_WEIGHT = globals().get('REPORT_AUX', 0.0) if globals().get('AUX_DIM', 0) else 0.0


def report_aux_loss(pred, target, known):
    # Squared error to the report embedding (components scaled to mean variance 1), rows with one only.
    rows = known.all(1)
    if not rows.any():
        return pred.sum() * 0
    return ((pred[rows].float() - target[rows]) ** 2).mean()


all_hist = []
for fold in FOLDS_TO_RUN:
    assert fold in range(N_FOLDS)
    fold_seed = SEED + fold
    torch.manual_seed(fold_seed); np.random.seed(fold_seed); random.seed(fold_seed)
    # Gold studies never validate; with GOLD_WEIGHT > 0 they train in every fold.
    tr_rows = np.flatnonzero(((folds >= 0) & (folds != fold)) | ((folds == -1) & (GOLD_WEIGHT > 0)))
    va_rows = np.flatnonzero(folds == fold)
    assert not set(tr_rows) & set(va_rows)
    assert df.iloc[va_rows]['source'].eq('pseudo').all()
    assert GOLD_WEIGHT > 0 or df.iloc[tr_rows]['source'].eq('pseudo').all()
    dl_tr = make_loader(tr_rows, True, fold_seed)
    dl_val = make_loader(va_rows, False, fold_seed)
    model = build_model(recipe, len(LABEL_COLS), len(PLANES),
                        pretrained=not (RESUME and os.path.exists(os.path.join(OUT, f'last_fold_{fold}.pt')))).to(DEV)
    if GRAD_CHECKPOINT and hasattr(model.enc, 'set_grad_checkpointing'):
        model.enc.set_grad_checkpointing(True)     # recompute activations: fits large backbones on a T4
    # Frozen encoder blocks are left out of the optimizer entirely.
    encoder_params = [p for p in model.enc.parameters() if p.requires_grad]
    head_params = [p for name,p in model.named_parameters() if not name.startswith('enc.')]
    print(f'trainable: encoder {sum(p.numel() for p in encoder_params)/1e6:.1f}M, '
          f'head {sum(p.numel() for p in head_params)/1e6:.2f}M')
    opt = torch.optim.AdamW([{'params':encoder_params,'lr':LR*ENCODER_LR_SCALE},
                            {'params':head_params,'lr':LR}], weight_decay=WD)
    total_steps = EPOCHS*math.ceil(len(dl_tr)/ACCUM_STEPS)
    pct_start = PCT_START
    if abs(pct_start * total_steps - 1) < 1e-6:
        # OneCycleLR divides by zero when the warmup is exactly one optimizer step.
        pct_start = 1.5 / total_steps if total_steps >= 2 else .5
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR*ENCODER_LR_SCALE, LR], total_steps=total_steps,
                    pct_start=pct_start)
    scaler = torch.amp.GradScaler(DEV, enabled=DEV=='cuda')
    # Exponential moving average of the weights; it is what validates and what is saved.
    ema, ema_steps = None, 0
    if EMA_DECAY:
        ema = copy.deepcopy(model).eval()
        for p in ema.parameters(): p.requires_grad_(False)
    path = os.path.join(OUT, f'fold_{fold}.pt')
    last_path = os.path.join(OUT, f'last_fold_{fold}.pt')
    start, best, hist = 0, float('inf'), []
    if RESUME and os.path.exists(last_path):
        saved = torch.load(last_path, map_location='cpu', weights_only=False)
        assert saved['cfg'] == recipe, 'Resume recipe differs; use a new OUT directory'
        model.load_state_dict(saved['model']); opt.load_state_dict(saved['optimizer'])
        sched.load_state_dict(saved['scheduler']); scaler.load_state_dict(saved['scaler'])
        start, best, hist = saved['epoch']+1, saved['best'], saved['history']
        if ema is not None:
            ema.load_state_dict(saved['ema']); ema_steps = saved['ema_steps']
        assert os.path.exists(path), 'Resume requires both last and best fold checkpoints'
    elif os.path.exists(path):
        raise RuntimeError(f'{path} already exists; set RESUME=True or choose an empty OUT')
    for ep in range(start, EPOCHS):
        # Epoch-local worker seeds make interrupted runs repeatable at epoch boundaries.
        epoch_seed = fold_seed*1000 + ep
        torch.manual_seed(epoch_seed); np.random.seed(epoch_seed); random.seed(epoch_seed)
        dl_tr = make_loader(tr_rows, True, epoch_seed)
        model.train(); total, count, aux_total = 0., 0, 0.; t0 = time.time()
        opt.zero_grad(set_to_none=True)
        for step, (x, present, y, w) in enumerate(tqdm(dl_tr, desc=f'fold {fold} epoch {ep+1}')):
            x, present, y, w = x.to(DEV, non_blocking=True), present.to(DEV), y.to(DEV), w.to(DEV)
            n_lab = len(LABEL_COLS)
            y_aux, w_aux, y, w = y[:, n_lab:], w[:, n_lab:], y[:, :n_lab], w[:, :n_lab]
            logits, _ = model(x, present)
            loss = masked_bce(logits.float(), y, w)
            if REPORT_AUX_WEIGHT:
                aux = report_aux_loss(model.last_aux, y_aux, w_aux > 0)
                loss = loss + REPORT_AUX_WEIGHT * aux
                aux_total += aux.item() * len(x)
            if not torch.isfinite(loss): raise FloatingPointError('Non-finite loss')
            group_start = (step // ACCUM_STEPS) * ACCUM_STEPS
            group_size = min(ACCUM_STEPS, len(dl_tr)-group_start)
            scaler.scale(loss/group_size).backward()
            if (step+1) % ACCUM_STEPS == 0 or step+1 == len(dl_tr):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP)
                old_scale = scaler.get_scale()
                scaler.step(opt); scaler.update()
                if scaler.get_scale() >= old_scale: sched.step()
                opt.zero_grad(set_to_none=True)
                if ema is not None:
                    ema_steps += 1
                    decay = min(EMA_DECAY, (1 + ema_steps) / (10 + ema_steps))
                    with torch.no_grad():
                        for pe, pm in zip(ema.parameters(), model.parameters()): pe.lerp_(pm, 1 - decay)
                        for be, bm in zip(ema.buffers(), model.buffers()): be.copy_(bm)
            total += loss.item()*len(x); count += len(x)
        judged = ema if ema is not None else model
        yv, pv = evaluate(judged, dl_val)
        observed = np.isfinite(yv)
        clipped = np.clip(pv, 1e-7, 1-1e-7)
        val_loss = float((-(np.nan_to_num(yv)*np.log(clipped) +
                           (1-np.nan_to_num(yv))*np.log1p(-clipped)))[observed].mean())
        auc, _ = macro_auc(yv, pv, LABEL_COLS)
        rec = dict(fold=fold, epoch=ep, loss=total/count, val_bce=val_loss,
                   auc_pseudo=auc, minutes=(time.time()-t0)/60)
        if REPORT_AUX_WEIGHT:
            rec['report_aux'] = aux_total / count
        hist.append(rec); print(rec)
        # Soft-label BCE on pseudo-labelled validation selects epochs; gold is never consulted.
        if val_loss < best:
            best = val_loss
            atomic_torch_save(dict(model=judged.state_dict(), cfg=recipe, epoch=ep,
                                   fold=fold, val_bce=best, auc_pseudo=auc), path)
        atomic_torch_save(dict(model=model.state_dict(), cfg=recipe, epoch=ep, best=best,
                               optimizer=opt.state_dict(), scheduler=sched.state_dict(),
                               scaler=scaler.state_dict(), history=hist,
                               ema=None if ema is None else ema.state_dict(), ema_steps=ema_steps), last_path)
        pd.DataFrame(hist).to_csv(os.path.join(OUT, f'history_fold_{fold}.csv'), index=False)
    model.load_state_dict(torch.load(path, map_location=DEV, weights_only=False)['model'])
    yv, pv = evaluate(model, dl_val)
    output = pd.DataFrame(pv, columns=LABEL_COLS)
    output.insert(0,'StudyInstanceUID',df.iloc[va_rows]['StudyInstanceUID'].to_numpy())
    output['fold'] = fold
    output.to_csv(os.path.join(OUT,f'oof_fold_{fold}.csv'), index=False)
    gold_rows = np.flatnonzero(folds == -1)
    yg, pg = evaluate(model, make_loader(gold_rows, False, fold_seed))
    gold_output = pd.DataFrame(pg, columns=LABEL_COLS)
    gold_output.insert(0,'StudyInstanceUID',df.iloc[gold_rows]['StudyInstanceUID'].to_numpy())
    gold_output.to_csv(os.path.join(OUT,f'gold_fold_{fold}.csv'), index=False)
    print('Gold audit after checkpoint selection'
          + (' (IN-SAMPLE: gold trained at weight %g)' % GOLD_WEIGHT if GOLD_WEIGHT > 0 else '') + ':',
          macro_auc(yg,pg,LABEL_COLS)[0])
    all_hist.extend(hist)
    del model, ema, opt, sched, scaler, dl_tr, dl_val
    gc.collect()
    if DEV=='cuda': torch.cuda.empty_cache()

paths = [os.path.join(OUT,f'oof_fold_{f}.csv') for f in range(N_FOLDS)]
if all(os.path.exists(p) for p in paths):
    oof = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
    expected = df.loc[folds>=0,'StudyInstanceUID']
    assert oof['StudyInstanceUID'].is_unique and set(oof['StudyInstanceUID']) == set(expected)
    aligned = oof.set_index('StudyInstanceUID').loc[expected]
    oof.to_csv(os.path.join(OUT,'oof_predictions.csv'), index=False)
    print('OOF pseudo AUC (noisy-label proxy, selection-biased):',
          macro_auc(df.loc[folds>=0,LABEL_COLS].to_numpy(),aligned[LABEL_COLS].to_numpy(),LABEL_COLS))
else:
    print('Partial fold run; complete all folds before comparing aggregate OOF results.')

# Fold predictions share the same gold study order; audit the exact inference blend.
gold_paths = [os.path.join(OUT,f'gold_fold_{f}.csv') for f in range(N_FOLDS)]
if all(os.path.exists(p) for p in gold_paths):
    gold_ids = df.loc[folds==-1,'StudyInstanceUID']
    if len(gold_ids):
        members = [pd.read_csv(p).set_index('StudyInstanceUID').loc[gold_ids,LABEL_COLS].to_numpy()
                   for p in gold_paths]
        blended = rank_blend(members)
        audit = pd.DataFrame(blended,columns=LABEL_COLS)
        audit.insert(0,'StudyInstanceUID',gold_ids.to_numpy())
        audit.to_csv(os.path.join(OUT,'gold_ensemble.csv'),index=False)
        print('Gold fold-ensemble audit' + (' (in-sample)' if GOLD_WEIGHT > 0 else '') + ':', macro_auc(df.loc[folds==-1,LABEL_COLS].to_numpy(),blended,LABEL_COLS))

## 5 · Per-finding results

In [ ]:
# Run cells in order; no working-directory changes are needed.

In [ ]:
# Results are saved per fold: histories, OOF predictions, and untouched gold audit predictions.

In [ ]:
# Results are saved per fold: histories, OOF predictions, and untouched gold audit predictions.

## 6 · Optional: throughput check

Run this if training feels slow. It separates data from compute, which is the only
way to tell them apart — guessing cost a wasted run once already.

In [ ]:
# Results are saved per fold: histories, OOF predictions, and untouched gold audit predictions.

## Next

Save this executed output containing `fold_2.pt`, `last_fold_2.pt`, history and audit CSVs, then attach it to 05_submit.ipynb. Folds run independently; a family may contribute fewer than five folds.